In [9]:
import requests
from PIL import Image
import gradio as gr
import io

# Custom Vision Prediction 키와 URL


# Gradio 인터페이스 정의



In [ ]:
import requests                  # HTTP 요청을 보내기 위한 라이브러리
from PIL import Image            # 이미지 처리용 라이브러리 (Pillow)
import gradio as gr              # Gradio UI 생성 라이브러리
import io                        # 바이트 스트림 처리를 위한 모듈

# Custom Vision Prediction 서비스 키와 엔드포인트 URL 설정
PREDICTION_KEY = ""
ENDPOINT_URL   = ""
# API 호출에 사용할 헤더, Pr

# API 호출에 사용할 헤더, Prediction-Key와 데이터 타입 지정
headers = {
    "Prediction-Key": PREDICTION_KEY,
    "Content-Type": "application/octet-stream"   # 바이너리 이미지 데이터 전송
}

In [11]:
# 예측 함수 정의 (Gradio에 연결할 함수)
def predict_with_api(image: Image.Image):
    # 이미지를 바이너리로 변환
    buf = io.BytesIO()                          # 메모리 내 바이트 버퍼 생성
    image.save(buf, format='JPEG')              # 이미지를 JPEG 형식으로 저장
    byte_data = buf.getvalue()                  # 버퍼에서 바이너리 데이터 추출

    # Azure Custom Vision API에 POST 요청
    response = requests.post(ENDPOINT_URL, headers=headers, data=byte_data)

    # API 응답에서 예측 결과 추출 (JSON 형식)
    predictions = response.json()["predictions"]

    # 확률이 높은 상위 2개 결과만 선택 (내림차순 정렬)
    top_predictions = sorted(predictions, key=lambda x: x["probability"], reverse=True)[:2]

    # 결과 문자열 포맷팅
    result_lines = [
        f"{pred['tagName']} ({pred['probability'] * 100:.2f}%)"  # 예: "cat (97.23%)"
        for pred in top_predictions
    ]

    return "\n".join(result_lines)  # 여러 줄의 문자열로 반환

In [4]:
# Gradio 인터페이스 정의
interface = gr.Interface(
    fn=predict_with_api,                   # 호출할 함수
    inputs=gr.Image(type="pil"),           # 사용자 입력: 이미지 (PIL 형식)
    outputs=gr.Text(),                     # 출력 형식: 텍스트
    title="Custom Vision Image Classifier",   # 웹 앱 제목
    description="Upload an image to see the prediction from your Custom Vision model."  # 설명 텍스트
)

# Gradio 웹 인터페이스 실행
interface.launch()

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


In [12]:
import io
import requests
import gradio as gr
from PIL import Image

# ===== 예측 함수 (인자 4개, 반환 2개) =====
def predict_with_api(image: Image.Image, top_k: int, show_prob: bool, threshold: float):
    # 입력 방어
    if image is None:
        return "이미지를 업로드해 주세요.", {}

    # PIL → JPEG bytes
    if image.mode != "RGB":
        image = image.convert("RGB")
    buf = io.BytesIO()
    image.save(buf, format="JPEG")
    byte_data = buf.getvalue()

    # API 호출
    try:
        response = requests.post(ENDPOINT_URL, headers=headers, data=byte_data, timeout=15)
    except Exception as e:
        return f"요청 실패: {type(e).__name__}: {e}", {}

    if response.status_code != 200:
        return f"API 에러 {response.status_code}: {response.text[:200]}", {}

    try:
        result = response.json()
    except Exception as e:
        return f"JSON 파싱 실패 ({type(e).__name__}). 응답: {response.text[:200]}", {}

    predictions = result.get("predictions", [])
    if not predictions:
        return "예측 결과가 없습니다.", {}

    # 임계값 필터링
    filtered = [p for p in predictions if p["probability"] >= threshold]
    if not filtered:
        return f"임계값({threshold:.2f}) 이상의 예측이 없습니다.", {}

    # 상위 K개 정렬
    top_predictions = sorted(filtered, key=lambda x: x["probability"], reverse=True)[:top_k]

    # 텍스트 결과
    if show_prob:
        result_lines = [
            f"{p['tagName']} ({p['probability'] * 100:.2f}%)"
            for p in top_predictions
        ]
    else:
        result_lines = [p["tagName"] for p in top_predictions]
    text_output = "\n".join(result_lines)

    # gr.Label용 dict (라벨 → 확률)
    label_output = {p["tagName"]: float(p["probability"]) for p in top_predictions}

    return text_output, label_output


# ===== Gradio UI =====
with gr.Blocks() as demo:
    gr.Markdown("## 🧠 Azure Custom Vision Classifier")
    gr.Markdown("Upload an image and configure settings to get predictions from your trained model.")

    with gr.Row():
        with gr.Column():
            image_input = gr.Image(type="pil", label="Input Image")
            top_k_dropdown = gr.Dropdown([1, 2, 3, 5], value=2, label="Top-K Predictions")
            prob_checkbox = gr.Checkbox(label="Show Prediction Probabilities", value=True)
            threshold_slider = gr.Slider(minimum=0.0, maximum=1.0, value=0.5, step=0.05,
                                         label="Confidence Threshold")
            predict_btn = gr.Button("🔍 Predict")
        with gr.Column():
            output_text = gr.Textbox(label="Prediction Text", lines=5)
            output_label = gr.Label(label="Prediction Scores")

    predict_btn.click(
        fn=predict_with_api,
        inputs=[image_input, top_k_dropdown, prob_checkbox, threshold_slider],
        outputs=[output_text, output_label],
    )

demo.launch()

* Running on local URL:  http://127.0.0.1:7866
* To create a public link, set `share=True` in `launch()`.
